<a href="https://colab.research.google.com/github/xiaoWings/cocktail-recipe-finder/blob/main/Investigation_Notebook_Group22_Cocktail_Recipe_Finder_Webapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Goals**


In this notebook we will fetch and process coctail recipe data from TheCocktailDB, a public open api source.
Specifically I want to explore the feasibility of search coctail by name, seach cocktail recipe by liquor name, Search liquor / ingredient metadata, Get full recipe, Ingredient lookup by ID, List ingredients/categories/glasses/alcohol filters.



# **Reference**

### TheCoctailDB API
https://www.thecocktaildb.com/documentation

Base URL: www.thecocktaildb.com/api/json/v1/APIKEY/

Search cocktail by name:
/api/json/v1/1/search.php?s=margarita

Search cocktail by first letter:
/api/json/v1/1/search.php?f=a

Search ingredient by name:
/api/json/v1/1/search.php?i=vodka

Lookup full cocktail details by id:
/api/json/v1/1/lookup.php?i=11007

Lookup ingredient by id:
/api/json/v1/1/lookup.php?iid=552

Lookup a random cocktail:
/api/json/v1/1/random.php

Search by ingredient:

/api/json/v1/1/filter.php?i=Gin

/api/json/v1/1/filter.php?i=Vodka

Filter by alcoholic category:

/api/json/v1/1/filter.php?a=Alcoholic

/api/json/v1/1/filter.php?a=Non_Alcoholic

Filter by category:

/api/json/v1/1/filter.php?c=Ordinary_Drink

/api/json/v1/1/filter.php?c=Cocktail

Filter by glass:

/api/json/v1/1/filter.php?g=Cocktail_glass

/api/json/v1/1/filter.php?g=Champagne_flute

List categories, glasses, ingredients, or alcoholic filters:
/api/json/v1/1/list.php?c=list

/api/json/v1/1/list.php?g=list

/api/json/v1/1/list.php?i=list

/api/json/v1/1/list.php?a=list




# API Moduler Testing

###  1.Relevant API endpoints and how they will support my workflow
| Project need                                        | Best endpoint                                                              | Role in your app                                                                                                                                                                                                                                                                               |
| --------------------------------------------------- | -------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Search cocktail by name                             | `search.php?s=margarita`                                                   | Lets users search a known cocktail name, useful for testing and manual lookup. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation)                                                                                                                     |
| Search liquor / ingredient metadata                 | `search.php?i=vodka`                                                       | Returns ingredient details; useful for ingredient validation or info display. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation), [\[thecocktaildb.com\]](https://www.thecocktaildb.com/AGENTS.md)                                                        |
| Find cocktails containing one ingredient or liquor  | `filter.php?i=Gin`                                                         | Main discovery endpoint for “I have gin/vodka/tequila.” Use this for both ingredient and liquor workflows. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation), [\[thecocktaildb.com\]](https://www.thecocktaildb.com/AGENTS.md)                           |
| Get full recipe                                     | `lookup.php?i=11007`                                                       | Required after discovery because it provides full recipe details, ingredients, measures, and instructions. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation), [\[thecocktaildb.com\]](https://www.thecocktaildb.com/AGENTS.md)                           |
| Ingredient lookup by ID                             | `lookup.php?iid=552`                                                       | Optional enrichment if you want ingredient detail pages. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation), [\[thecocktaildb.com\]](https://www.thecocktaildb.com/AGENTS.md)                                                                             |
| Random cocktail                                     | `random.php`                                                               | Good for testing API connectivity and “surprise me” feature. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation), [\[thecocktaildb.com\]](https://www.thecocktaildb.com/AGENTS.md)                                                                         |
| List ingredients/categories/glasses/alcohol filters | `list.php?i=list`, `list.php?c=list`, `list.php?g=list`, `list.php?a=list` | Useful for dropdowns, autocomplete, filters, and cleaner user inputs. [\[thecocktaildb.com\]](https://www.thecocktaildb.com/documentation), [\[thecocktaildb.com\]](https://www.thecocktaildb.com/AGENTS.md)                                                                |


### 2.Key API design finding: how ingredient data is represented
Full drink results are returned inside a top-level drinks property, and useful recipe fields include idDrink, strDrink, strCategory, strAlcoholic, strGlass, strInstructions, strDrinkThumb, strIngredient1 through strIngredient15, and strMeasure1 through strMeasure15. The numbered ingredient and measure fields correspond by number, so strIngredient3 should be paired with strMeasure3; TheCocktailDB guidance says to trim whitespace, ignore null or empty ingredient slots, preserve order, and expect that a measure may be blank even when an ingredient exists.

Parsing logic

In [ ]:
def extract_ingredients(drink):
    """
    Converts TheCocktailDB's flat ingredient/measure fields
    into a cleaner list of dictionaries.
    """
    ingredients = []

    for i in range(1, 16):
        ingredient = drink.get(f"strIngredient{i}")
        measure = drink.get(f"strMeasure{i}")

        if ingredient and ingredient.strip():
            ingredients.append({
                "ingredient": ingredient.strip(),
                "measure": measure.strip() if measure else ""
            })

    return ingredients

### **3. Feasibility code examples**

   #### 3.1 Setup and reusable API helper

In [ ]:
import requests
from urllib.parse import urlencode

BASE_URL = "https://www.thecocktaildb.com/api/json/v1/1"

def get_json(endpoint, params=None):
    """
    Calls TheCocktailDB API and returns parsed JSON.
    """
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

In [ ]:
import requests
import pandas as pd
from IPython.display import display, Image, Markdown
from collections import defaultdict
import re

BASE_URL = "https://www.thecocktaildb.com/api/json/v1/1"

print("Setup complete.")
print("Base URL:", BASE_URL)

Setup complete.
Base URL: https://www.thecocktaildb.com/api/json/v1/1


#### 3.2 Test: search cocktail by name

In [ ]:
# ask user to input a cocktail name and return the first drink record's info
while True:
    cocktail_name = input("Enter a cocktail name to search (or type 'quit' to exit): ")
    if cocktail_name.lower() in ('quit', 'exit'):
        print("Exiting search.")
        break

    data = get_json("search.php", {"s": cocktail_name})

    drinks = data.get("drinks") or []
    print("Number of drinks returned:", len(drinks))

    if drinks:
        first = drinks[0]
        print(first["idDrink"], first["strDrink"])
        print(first["strCategory"])
        print(first["strAlcoholic"])
        print(first["strGlass"])
        print(first["strInstructions"][:200])
    else:
        print(f"No drinks found with the name '{cocktail_name}'.")
    print("---\n")

Enter a cocktail name to search (or type 'quit' to exit): rye
Number of drinks returned: 0
No drinks found with the name 'rye'.
---

Enter a cocktail name to search (or type 'quit' to exit): manhattan
Number of drinks returned: 1
11008 Manhattan
Cocktail
Alcoholic
Cocktail glass
Stirred over ice, strained into a chilled glass, garnished, and served up.
---

Enter a cocktail name to search (or type 'quit' to exit): quit
Exiting search.


This proves the API returns drink records as JSON and provides fields needed for a recipe display.

#### 3.3 Test: filter by liquor

In [ ]:
while True:
    liquor_name = input("Enter a liquor name to search (or type 'quit' to exit): ")
    if liquor_name.lower() in ('quit', 'exit'):
        print("Exiting liquor search.")
        break

    print(f"Searching for cocktails with '{liquor_name}'...")
    data_filter = get_json("filter.php", {"i": liquor_name})
    candidates = data_filter.get("drinks") or []

    if not candidates:
        print(f"No cocktails found containing '{liquor_name}'.")
    else:
        print(f"Found {len(candidates)} candidate cocktails with '{liquor_name}'.")
        for i, candidate_drink in enumerate(candidates):
            drink_id = candidate_drink["idDrink"]
            drink_name = candidate_drink["strDrink"]
            print(f"\n--- Cocktail {i+1} of {len(candidates)} ---")
            print(f"ID: {drink_id}, Name: {drink_name}")

            # Lookup full recipe details
            data_lookup = get_json("lookup.php", {"i": drink_id})
            full_drink_details = data_lookup.get("drinks")

            if full_drink_details:
                drink = full_drink_details[0]
                print(f"Category: {drink['strCategory']}")
                print(f"Alcoholic: {drink['strAlcoholic']}")
                print(f"Glass: {drink['strGlass']}")
                print(f"Instructions: {drink['strInstructions']}")

                ingredients = extract_ingredients(drink)
                print("Ingredients:")
                for ing in ingredients:
                    print(f"  - {ing['measure'].ljust(15)} {ing['ingredient']}")
            else:
                print(f"Could not retrieve full details for {drink_name} (ID: {drink_id}).")
    print("\n========================================\n")

Enter a liquor name to search (or type 'quit' to exit): whiskey
Searching for cocktails with 'whiskey'...
Found 1 candidate cocktails with 'whiskey'.

--- Cocktail 1 of 1 ---
ID: 13194, Name: Damned if you do
Category: Shot
Alcoholic: Alcoholic
Glass: Shot glass
Instructions: Pour into shot glass. Put in mouth. Repeat as deemed necessary.
Ingredients:
  - 0.75 oz         Whiskey
  - 0.25 oz         Hot Damn


Enter a liquor name to search (or type 'quit' to exit): mezcal
Searching for cocktails with 'mezcal'...
Found 1 candidate cocktails with 'mezcal'.

--- Cocktail 1 of 1 ---
ID: 17246, Name: Empellón Cocina's Fat-Washed Mezcal
Category: Cocktail
Alcoholic: Alcoholic
Glass: Beer Glass
Instructions: To ensure that your pork fat is just as delicious as theirs, here’s their adobo marinade and what to do with it (you’ll also need a rack of ribs):

4 ancho chiles, 8 guajillo chiles and 4 chipotle chiles, plus 4 cloves roasted garlic, half a cup of cider vinegar, a quarter teaspoon of Mexi

#### 3.4 Test: lookup full cocktail details by ID

In [ ]:
while True:
    ingredient_name = input("\nEnter an ingredient name to search (or type 'quit' to exit): ").strip()
    if ingredient_name.lower() in ('quit', 'exit'):
        print("Exiting ingredient search.")
        break

    print(f"Searching for cocktails containing '{ingredient_name}'...")
    data_filter = get_json("filter.php", {"i": ingredient_name})

    # Check if 'drinks' is a list; otherwise, treat it as empty
    raw_drinks_data = data_filter.get("drinks")
    if isinstance(raw_drinks_data, list):
        candidates = raw_drinks_data
    else:
        candidates = [] # Handle 'no data found' string or None

    if not candidates:
        print(f"No cocktails found containing '{ingredient_name}'.")
    else:
        print(f"Found {len(candidates)} cocktails:")
        for i, drink_summary in enumerate(candidates):
            print(f"  {i+1}. {drink_summary['strDrink']}")

        while True:
            try:
                choice = input("\nEnter the number of the cocktail to see the full recipe, or 'back' to search another ingredient: ").strip()
                if choice.lower() == 'back':
                    break # Go back to the ingredient search loop

                choice_index = int(choice) - 1
                if 0 <= choice_index < len(candidates):
                    selected_drink_id = candidates[choice_index]["idDrink"]
                    selected_drink_name = candidates[choice_index]["strDrink"]

                    print(f"Fetching full recipe for {selected_drink_name}...")
                    detail_data = get_json("lookup.php", {"i": selected_drink_id})
                    full_drink_details = detail_data.get("drinks")

                    if full_drink_details:
                        drink = full_drink_details[0]
                        print(f"\n--- Full Recipe for {drink['strDrink']} (ID: {drink['idDrink']}) ---")
                        print(f"Category: {drink['strCategory']}")
                        print(f"Alcoholic: {drink['strAlcoholic']}")
                        print(f"Glass: {drink['strGlass']}")
                        print(f"Instructions: {drink['strInstructions']}")

                        ingredients = extract_ingredients(drink)
                        print("Ingredients:")
                        for ing in ingredients:
                            print(f"  - {ing['measure'].ljust(15)} {ing['ingredient']}")
                    else:
                        print(f"Could not retrieve full details for the selected cocktail (ID: {selected_drink_id}).")

                    another_choice = input("\nView another cocktail from this list (yes/no)? ").strip().lower()
                    if another_choice != 'yes':
                        break # Exit inner loop, go back to ingredient search
                else:
                    print("Invalid choice. Please enter a valid number or 'back'.")
            except ValueError:
                print("Invalid input. Please enter a number or 'back'.")
            except Exception as e:
                print(f"An unexpected error occurred: {e}")
    print("\n========================================\n") # Separator for new ingredient search


Enter an ingredient name to search (or type 'quit' to exit): lemon
Searching for cocktails containing 'lemon'...
Found 1 cocktails:
  1. 3-Mile Long Island Iced Tea

Enter the number of the cocktail to see the full recipe, or 'back' to search another ingredient: 1
Fetching full recipe for 3-Mile Long Island Iced Tea...

--- Full Recipe for 3-Mile Long Island Iced Tea (ID: 15300) ---
Category: Ordinary Drink
Alcoholic: Alcoholic
Glass: Collins Glass
Instructions: Fill 14oz glass with ice and alcohol. Fill 2/3 glass with cola and remainder with sweet & sour. Top with dash of bitters and lemon wedge.
Ingredients:
  - 1/2 oz          Gin
  - 1/2 oz          Light rum
  - 1/2 oz          Tequila
  - 1/2 oz          Triple sec
  - 1/2 oz          Vodka
  - 1/2 oz          Coca-Cola
  - 1-2 dash        Sweet and sour
  - 1 wedge         Bitters
  - Garnish with    Lemon

View another cocktail from this list (yes/no)? yes

Enter the number of the cocktail to see the full recipe, or 'back' to 

#### 3.5 Compare recipe ingredients against ingredients on hand

In [ ]:
def normalize(text):
    """
    Basic normalization for matching user inputs to recipe ingredients.
    """
    return text.strip().lower()

def compare_to_pantry(recipe_ingredients, user_pantry):
    """
    Returns available and missing ingredients for one cocktail.
    """
    normalized_pantry = {normalize(item) for item in user_pantry}

    available = []
    missing = []

    for item in recipe_ingredients:
        ingredient_name = item["ingredient"]
        normalized_name = normalize(ingredient_name)

        if normalized_name in normalized_pantry:
            available.append(item)
        else:
            missing.append(item)

    return available, missing


# Main interaction loop
while True:
    user_pantry_input = input("\nEnter the ingredients you have, separated by commas (or type 'quit' to exit): ").strip()
    if user_pantry_input.lower() in ('quit', 'exit'):
        print("Exiting cocktail suggestion tool.")
        break

    user_pantry_list = [ing.strip() for ing in user_pantry_input.split(',') if ing.strip()]

    if not user_pantry_list:
        print("Please enter at least one ingredient.")
        continue

    print(f"Searching for cocktails with: {', '.join(user_pantry_list)}")

    # Prepare a list to store matching cocktails with missing ingredient count
    matching_cocktails = []

    # Iterate through the DataFrame of all cocktails (df_cocktails is assumed to be loaded)
    for index, row in df_cocktails.iterrows():
        drink_data = row.to_dict()
        # The extract_ingredients function expects a dictionary like the API response
        recipe_ingredients = extract_ingredients(drink_data)

        available_ing, missing_ing = compare_to_pantry(recipe_ingredients, user_pantry_list)

        matching_cocktails.append({
            "idDrink": drink_data["idDrink"],
            "strDrink": drink_data["strDrink"],
            "recipe_ingredients": recipe_ingredients,
            "available_ingredients": available_ing,
            "missing_ingredients": missing_ing,
            "num_missing": len(missing_ing)
        })

    # Sort cocktails by the number of missing ingredients (ascending)
    matching_cocktails.sort(key=lambda x: x["num_missing"])

    # Filter to only show drinks where at least one ingredient is available (optional, but good for relevance)
    # Or, perhaps, only show drinks that are not entirely missing
    # For this, let's keep all and just show the top 5

    if not matching_cocktails or all(c["num_missing"] == len(c["recipe_ingredients"]) for c in matching_cocktails):
        print("No suitable cocktails found based on your ingredients.")
    else:
        print("\n--- Top 5 Cocktail Suggestions ---")
        top_suggestions = matching_cocktails[:5]
        for i, cocktail in enumerate(top_suggestions):
            print(f"  {i+1}. {cocktail['strDrink']} (Missing {cocktail['num_missing']} ingredient(s))")

        while True:
            try:
                choice = input("\nEnter the number of the cocktail to see the full recipe, 'back' to change your ingredients, or 'quit' to exit: ").strip()
                if choice.lower() == 'back':
                    break # Go back to ingredient input
                if choice.lower() in ('quit', 'exit'):
                    print("Exiting cocktail suggestion tool.")
                    exit() # Exit the entire script

                choice_index = int(choice) - 1
                if 0 <= choice_index < len(top_suggestions):
                    selected_cocktail = top_suggestions[choice_index]
                    print(f"\n--- Full Recipe for {selected_cocktail['strDrink']} ---")

                    # Fetch full details again to get instructions etc., might be redundant if df_cocktails is comprehensive
                    # But safer to get fresh if df_cocktails isn't fully detailed on instructions
                    detail_data = get_json("lookup.php", {"i": selected_cocktail["idDrink"]})
                    full_drink_details = detail_data.get("drinks")

                    if full_drink_details:
                        drink_full = full_drink_details[0]
                        print(f"Category: {drink_full['strCategory']}")
                        print(f"Alcoholic: {drink_full['strAlcoholic']}")
                        print(f"Glass: {drink_full['strGlass']}")
                        print(f"Instructions: {drink_full['strInstructions']}")

                        print("Ingredients:")
                        for ing in selected_cocktail['available_ingredients']:
                            print(f"  ✅ {ing['measure'].ljust(15)} {ing['ingredient']} (You have this)")
                        for ing in selected_cocktail['missing_ingredients']:
                            print(f"  ❌ {ing['measure'].ljust(15)} {ing['ingredient']} (You need this)")
                    else:
                        print("Could not retrieve full details for the selected cocktail.")

                    another_recipe_choice = input("\nView another suggested recipe (yes/no), or 'back' to change ingredients: ").strip().lower()
                    if another_recipe_choice != 'yes':
                        break # Go back to suggestions or ingredient input
                else:
                    print("Invalid choice. Please enter a valid number, 'back', or 'quit'.")
            except ValueError:
                print("Invalid input. Please enter a number, 'back', or 'quit'.")
            except Exception as e:
                print(f"An unexpected error occurred: {e}")
    print("\n========================================\n")


Enter the ingredients you have, separated by commas (or type 'quit' to exit): whiskey, lemon, raspberry syrup
Searching for cocktails with: whiskey, lemon, raspberry syrup

--- Top 5 Cocktail Suggestions ---
  1. Damned if you do (Missing 1 ingredient(s))
  2. Royal Bitch (Missing 2 ingredient(s))
  3. Iced Coffee Fillip (Missing 2 ingredient(s))
  4. Ziemes Martini Apfelsaft (Missing 2 ingredient(s))
  5. A. J. (Missing 2 ingredient(s))

Enter the number of the cocktail to see the full recipe, 'back' to change your ingredients, or 'quit' to exit: 5

--- Full Recipe for A. J. ---
Category: Ordinary Drink
Alcoholic: Alcoholic
Glass: Cocktail glass
Instructions: Shake ingredients with ice, strain into a cocktail glass, and serve.
Ingredients:
  ❌ 1 1/2 oz        Applejack (You need this)
  ❌ 1 oz            Grapefruit juice (You need this)

View another suggested recipe (yes/no), or 'back' to change ingredients: 4



Enter the ingredients you have, separated by commas (or type 'quit' to

#### 3.6 Generate a grocery list for one or more selected cocktails

In [ ]:
from collections import defaultdict

def build_grocery_list(selected_drinks, user_ingredients):
    """
    selected_drinks: list of full drink dictionaries from lookup.php
    user_ingredients: list of ingredient names user already owns
    """
    grocery = defaultdict(list)

    for drink in selected_drinks:
        recipe_items = extract_ingredients(drink)
        available, missing = compare_to_pantry(recipe_items, user_ingredients)

        for item in missing:
            key = normalize(item["ingredient"])
            grocery[key].append({
                "ingredient": item["ingredient"],
                "measure": item["measure"],
                "cocktail": drink["strDrink"]
            })

    return grocery


# Example using one selected drink:
grocery_list = build_grocery_list([drink], user_ingredients)

for ingredient_key, entries in grocery_list.items():
    print("\n", entries[0]["ingredient"])
    for entry in entries:
        print(f"  - {entry['measure']} for {entry['cocktail']}")


 Dark rum
  - 1 shot for 155 Belmont

 Light rum
  - 2 shots for 155 Belmont

 Vodka
  - 1 shot for 155 Belmont

 Orange juice
  - 1 shot for 155 Belmont


# Proposed Application Architecture

Architecture components
1. User interface
*   text input for ingredients on hand.
*   Optional dropdown populated from list.php?i=list.
*   Search mode: “ingredients on hand,” “liquor/spirit on hand,” or “cocktail name.”

2. API client
*   Reusable Python functions using requests.get.
*   URL encoding for user-provided values, as recommended by TheCocktailDB guidance.
thecocktaildb

3. Discovery layer
* Use filter.php?i=... for ingredient/liquor discovery.
* Use search.php?s=... for cocktail name lookup.
* Use random.php for testing or bonus feature.
4. Recipe detail layer
* Call lookup.php?i={idDrink} for each candidate.
* Parse ingredients and measures.
5. Matching engine
* Normalize user ingredients and recipe ingredients.
* Count available vs. missing ingredients.
* Sort cocktails by “fewest missing ingredients.”
6. Output layer
* Recipe cards with name, image, glass, instructions, alcoholic status, ingredients, and missing items.
* Grocery list grouped by missing ingredient.
* The app should credit TheCocktailDB for recipe data and imagery because official guidance recommends attribution when publishing or displaying results.

# Web App Approach

| Option    | Strengths                                                                                                                                                                                              | Tradeoffs                                                                                                                                                                                    | Recommendation                                                              |
| --------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------- |
| Streamlit | Fast Python-first UI, ideal for interactive data apps, fewer web concepts, good for notebooks-to-app transition. [\[docs.streamlit.io\]](https://docs.streamlit.io/)            | Less control over custom routing/templates than Flask.                                                                                                                                       | **Best choice for this project.**                                           |
| Flask     | Lightweight web framework with routes, templates, request handling, and JSON response support. [\[flask.pall...ojects.com\]](https://flask.palletsprojects.com/en/stable/quickstart/) | Requires more HTML/templates/routing knowledge; more boilerplate for a beginner app. [\[flask.pall...ojects.com\]](https://flask.palletsprojects.com/en/stable/quickstart/) | Good stretch option, but less efficient for an Intro to Python deliverable. |


In [ ]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 51.8 MB/s eta 0:00:00


In [ ]:
# Minimal Steamlit prototype sketch

# streamlit_app.py

import streamlit as st
import requests
from collections import defaultdict

BASE_URL = "https://www.thecocktaildb.com/api/json/v1/1"

def get_json(endpoint, params=None):
    response = requests.get(f"{BASE_URL}/{endpoint}", params=params)
    response.raise_for_status()
    return response.json()

def extract_ingredients(drink):
    items = []
    for i in range(1, 16):
        ingredient = drink.get(f"strIngredient{i}")
        measure = drink.get(f"strMeasure{i}")
        if ingredient and ingredient.strip():
            items.append({
                "ingredient": ingredient.strip(),
                "measure": measure.strip() if measure else ""
            })
    return items

def normalize(text):
    return text.strip().lower()

def compare_to_pantry(recipe_items, pantry_items):
    pantry = {normalize(item) for item in pantry_items}
    available, missing = [], []

    for item in recipe_items:
        if normalize(item["ingredient"]) in pantry:
            available.append(item)
        else:
            missing.append(item)

    return available, missing

def lookup_drink(drink_id):
    data = get_json("lookup.php", {"i": drink_id})
    drinks = data.get("drinks") or []
    return drinks[0] if drinks else None

st.title("Cocktail Finder + Grocery List")

pantry_text = st.text_input(
    "Ingredients you have at home, separated by commas",
    "gin, lemon juice"
)

main_ingredient = st.text_input(
    "Main liquor or ingredient to search",
    "Gin"
)

if st.button("Find cocktails"):
    pantry_items = [item.strip() for item in pantry_text.split(",") if item.strip()]

    data = get_json("filter.php", {"i": main_ingredient})
    candidates = data.get("drinks") or []

    if not candidates:
        st.warning("No cocktails found. Try a different ingredient.")
    else:
        results = []

        for candidate in candidates[:10]:
            full_drink = lookup_drink(candidate["idDrink"])
            if not full_drink:
                continue

            recipe_items = extract_ingredients(full_drink)
            available, missing = compare_to_pantry(recipe_items, pantry_items)

            results.append({
                "drink": full_drink,
                "available": available,
                "missing": missing
            })

        results.sort(key=lambda x: len(x["missing"]))

        for result in results:
            drink = result["drink"]
            st.subheader(drink["strDrink"])
            st.image(drink["strDrinkThumb"], width=200)
            st.write("Glass:", drink["strGlass"])
            st.write("Alcoholic status:", drink["strAlcoholic"])
            st.write("Instructions:", drink["strInstructions"])

            st.write("Missing ingredients:")
            for item in result["missing"]:
                st.write(f"- {item['ingredient']} ({item['measure']})")

2026-08-09 07:37:19.898 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-09 07:37:20.155 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-09 07:37:20.156 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-09 07:37:20.157 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-09 07:37:20.159 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-09 07:37:20.159 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-09 07:37:20.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-09 07:37:20.162 Session state does not 

# Limitations, risks, and data-quality considerations

1. Multi-ingredient filtering is not free in the documented API. The official docs identify multi-ingredient filtering as Premium-only, so the MVP should filter by one main ingredient first and then perform matching in Python.
thecocktaildb
+1
2. Ingredient matching will be imperfect. The API may use exact ingredient names such as “Lime Juice,” while users may type “lime,” so the MVP should start with simple lowercase matching and list this as a future enhancement.
3. Measures are text fields. The API pairs strMeasureN with strIngredientN, but measures may be blank even when ingredients exist, so the grocery list should preserve recipe-specific measure text rather than summing quantities automatically.
thecocktaildb
4. Filter results are incomplete for recipe display. The recommended workflow is to use idDrink from discovery results and then call the lookup endpoint before presenting full recipe information.
thecocktaildb
5. Public release may need production/Premium access. The documentation says the test key 1 is available for development and educational use, while public app-store release requires a production key/Premium access.
thecocktaildb
6. Food safety/allergy limitations. TheCocktailDB guidance says the database is a recipe resource, not an allergen certification service, so an app should avoid making allergy or medical claims.


Scratch Work Section

In [ ]:
#I want to see the full database of the coctails and what data type existed
import pandas as pd
import string
import time

print("Attempting to fetch a comprehensive list of cocktails. This may take some time...")

all_drink_ids = set()
all_drinks_data = []

# Step 1: Gather all unique drink IDs by searching by first letter
for letter in string.ascii_lowercase:
    print(f"Fetching cocktails starting with '{letter}'...")
    data = get_json("search.php", {"f": letter})
    drinks_for_letter = data.get("drinks")

    if drinks_for_letter:
        for drink_summary in drinks_for_letter:
            all_drink_ids.add(drink_summary["idDrink"])
    time.sleep(0.1) # Be kind to the API

print(f"Found {len(all_drink_ids)} unique cocktail IDs.")

# Step 2: Fetch full details for each unique drink ID
for i, drink_id in enumerate(list(all_drink_ids)):
    if i % 50 == 0: # Print progress every 50 drinks
        print(f"Fetching details for drink {i+1}/{len(all_drink_ids)} (ID: {drink_id})...")
    data = get_json("lookup.php", {"i": drink_id})
    full_details = data.get("drinks")
    if full_details:
        all_drinks_data.append(full_details[0])
    time.sleep(0.1) # Be kind to the API

print(f"Successfully fetched full details for {len(all_drinks_data)} cocktails.")

# Step 3: Create a Pandas DataFrame
df_cocktails = pd.DataFrame(all_drinks_data)

print("\n--- DataFrame Info (Data Types and Non-Null Counts) ---")
df_cocktails.info()

print("\n--- First 5 Rows of the Cocktail Database ---")
print(df_cocktails.head())

Attempting to fetch a comprehensive list of cocktails. This may take some time...
Fetching cocktails starting with 'a'...
Fetching cocktails starting with 'b'...
Fetching cocktails starting with 'c'...
Fetching cocktails starting with 'd'...
Fetching cocktails starting with 'e'...
Fetching cocktails starting with 'f'...
Fetching cocktails starting with 'g'...
Fetching cocktails starting with 'h'...
Fetching cocktails starting with 'i'...
Fetching cocktails starting with 'j'...
Fetching cocktails starting with 'k'...
Fetching cocktails starting with 'l'...
Fetching cocktails starting with 'm'...
Fetching cocktails starting with 'n'...
Fetching cocktails starting with 'o'...
Fetching cocktails starting with 'p'...
Fetching cocktails starting with 'q'...
Fetching cocktails starting with 'r'...
Fetching cocktails starting with 's'...
Fetching cocktails starting with 't'...
Fetching cocktails starting with 'u'...
Fetching cocktails starting with 'v'...
Fetching cocktails starting with 'w'..